# quarrel Flight agent example

This notebook walks through an Agent round trip against a running `quarrel-server` Flight endpoint (`grpc://localhost:50051` by default). Install the SDK dependency with `python3 -m pip install -r pythonsdk/requirements.txt` and set `PYTHONPATH=pythonsdk` before starting the kernel.

Both methods use `DoPut` to submit an Arrow batch and `DoGet` with the request id as ticket to stream the response batches.

In [ ]:
from quarrel_flight import (
    EMBED_REQUEST_SCHEMA,
    GENERATE_REQUEST_SCHEMA,
    QuarrelFlightClient,
    embed_request_columns,
    generate_request_columns,
    schema_hash,
)

## 1. Check the transport and schema contract

The health action is a `DoAction` returning a small JSON body with readiness and the Arrow schema hashes. Compare those hashes with the pure-Python specs before sending batches.

In [ ]:
client = QuarrelFlightClient("grpc://localhost:50051")
health = client.health()
print(health)
print("generate.request hash:", schema_hash(GENERATE_REQUEST_SCHEMA))
print("embed.request hash:  ", schema_hash(EMBED_REQUEST_SCHEMA))

## 2. Inspect the exact request payloads

Request builders are pure Python: you can inspect the column mapping before pyarrow creates the batch. The generated batch always follows the normative field order in `api/flight-schema.md`.

In [ ]:
print(generate_request_columns("agent-1", [128000, 271], max_tokens=16))
print(embed_request_columns("agent-1-vectors", [128000, 271]))
print(GENERATE_REQUEST_SCHEMA.names())

## 3. Generate tokens

`generate()` uploads one `generate.request` batch to `/quarrel/v1/generate`, then reads the queued `generate.response` batches with `DoGet`. The final row carries `finished=True` and a non-null `finish_reason`.

In [ ]:
rows = client.generate(
    "agent-1",
    [128000, 271],
    max_tokens=32,
    temperature=0.8,
    top_p=0.95,
    top_k=40,
    stop_token_ids=[],
)
for row in rows:
    print(row["sequence"], row["token_id"], row["logprob"], row["finished"])

## 4. Embed token ids

`embed()` uploads one `embed.request` batch to `/quarrel/v1/embed` (one row per token id). The server returns a deterministic, L2-normalised 4-dimensional vector per token using the same backend as generation.

In [ ]:
vectors = client.embed("agent-1-vectors", [128000, 271, 42])
for row in vectors:
    embedding = row["embedding"]
    norm = sum(value * value for value in embedding) ** 0.5
    print(row["token_id"], [round(value, 4) for value in embedding], "norm=", round(norm, 6))

client.close()